# Giao diện người dùng tạo sinh khai báo

Trong bài học này, bạn sẽ định nghĩa một danh mục (catalog) gồm các khối xây dựng giao diện người dùng (UI) có thể tái sử dụng và để AI agent tự động lắp ghép chúng thành các giao diện phong phú - chẳng hạn như trang tổng quan (dashboard), danh sách chuyến bay,... - mà không cần phải lập trình thủ công từng bố cục.

## 📋 Mục tiêu học tập

1. **Hiểu về đặc tả A2UI** - Một chuẩn đặc tả GenUI khai báo được Google xây dựng với sự hợp tác của CopilotKit.
2. **Định nghĩa danh mục component** - Tạo các định nghĩa và trình kết xuất để agent có thể lắp ghép tại thời điểm chạy (runtime).
3. **Sử dụng schema động và tĩnh** - So sánh các bố cục do agent tự tạo với các mẫu đã được định nghĩa sẵn.

---

## 🚀 Những gì bạn sẽ xây dựng

Một giao diện trò chuyện nơi agent có thể tự động lắp ghép các bề mặt UI phong phú - bảng điều khiển (dashboard), biểu đồ và danh sách chuyến bay - từ một danh mục các thành phần có thể tái sử dụng:

> *"Hãy cho tôi xem bảng điều khiển doanh số với các số liệu về tổng doanh thu, khách hàng mới và tỷ lệ chuyển đổi."*

<img src="images/a2ui-dynamic-dashboard.png" style="width: 50%; display: block; margin: 0 auto;">

---

## 🧩 GenUI khai báo là gì?

GenUI khai báo cho phép bạn định nghĩa một tập hợp các khối xây dựng UI cơ bản, sau đó AI agent sẽ lắp ráp chúng thành các giao diện hoàn chỉnh. Agent chọn các thành phần từ danh mục, sắp xếp chúng vào một schema và liên kết dữ liệu thực tế để hiển thị kết quả.

Có ba thành phần chính để hệ thống này hoạt động:

* **Danh mục component**: Các khối UI cơ bản mà ứng dụng của bạn hỗ trợ, được chia thành hai phần:
  * **Định nghĩa**: Mô tả độc lập với nền tảng về tên, các thuộc tính (props) và mục đích của từng component.
  * **Renderer**: Cài đặt cụ thể theo nền tảng để biến các định nghĩa thành UI thực tế (ví dụ: các component React).
* **Schema**: Mô tả có cấu trúc về việc sử dụng các component nào, cách chúng lồng vào nhau và mối quan hệ giữa chúng.
* **Liên kết dữ liệu**: Các giá trị tại thời điểm chạy (runtime) sẽ lấp đầy schema bằng nội dung thực tế như chi tiết chuyến bay, số liệu kinh doanh hoặc bản ghi.

Một mô hình tư duy dễ hiểu là **trò chơi Lego**: **Danh mục** là hộp chứa các mảnh ghép, **schema** là cách chúng gắn kết với nhau, và **liên kết dữ liệu** là những chi tiết hoàn thiện ở bước cuối cùng. Agent sẽ lắp ráp giao diện một cách linh hoạt trong khi ứng dụng của bạn vẫn giữ được quyền kiểm soát về tính nhất quán, độ an toàn và chất lượng hiển thị.

<img src="images/a2ui-overview.png" style="width: 100%; display: block; margin: 0 auto;">

---

## ⚖️ Tại sao nên sử dụng GenUI khai báo?

GenUI kiểm soát hoạt động rất tốt cho các bề mặt giao diện có lưu lượng truy cập cao nhất, nơi sự chính xác tuyệt đối là yếu tố quan trọng. Nhưng đối với các trường hợp ở "đuôi dài" (như công cụ nội bộ, trường hợp ngoại lệ, mục tiêu đa dạng của người dùng), việc tự tay lập trình từng bố cục là không thể mở rộng.

Đó là lúc GenUI khai báo phát huy tác dụng: Agent lắp ráp UI từ một tập hợp các khối xây dựng cố định, mang lại sự linh hoạt nhưng không làm giảm đi tính an toàn hay nhất quán.

<img src="images/long-tail.png" style="width: 50%; display: block; margin: 0 auto;">

**Ưu điểm:**
- **Linh hoạt trong khuôn khổ:** Agent điều chỉnh giao diện mà không vượt ra ngoài hệ thống thiết kế component của bạn.
- **Tự mang component của bạn:** Bạn định nghĩa các khối cơ bản, agent quyết định cách kết hợp chúng.
- **Tiết kiệm công sức:** Định nghĩa danh mục một lần, tái sử dụng ở mọi nơi.
- **Đa nền tảng từ thiết kế:** Cùng một schema có thể hiển thị trên web, mobile, Slack và tin nhắn văn bản.
- **Tiết kiệm token:** Agent làm việc từ một từ vựng cố định thay vì tạo ra mã code lập trình tùy ý.

**Nhược điểm:**
- **Thiếu kiểm soát đến từng pixel:** Bạn không thể tinh chỉnh chi tiết bố cục cuối cùng do agent tạo ra.
- **Khó đoán trước:** Agent có thể lắp ráp các component theo những cách khác nhau trong các tình huống tương tự.
- **Dễ xảy ra lỗi logic:** Các schema và liên kết dữ liệu có thể bị lỗi ở những khía cạnh tinh vi, yêu cầu phải có logic kiểm tra và phục hồi.
- **Cần thiết kế kỹ từ đầu:** Danh mục component, định dạng schema, và cấu trúc renderer cần được định nghĩa cẩn thận.

---

## 🛠 Chuẩn bị môi trường

Đầu tiên, tải các thư viện cần thiết và thiết lập API key.

In [1]:
# Bỏ qua các cảnh báo không cần thiết
import warnings
warnings.filterwarnings("ignore")

import os
from helper import get_gemini_api_key, install_frontend

# Cài đặt các gói giao diện frontend (chạy npm install ngầm)
install_frontend()

# Nạp Google API Key
os.environ["GOOGLE_API_KEY"] = get_gemini_api_key()
print("✓ Google API key loaded")

Installing frontend dependencies ...

up to date, audited 971 packages in 5s

249 packages are looking for funding
  run `npm fund` for details

34 vulnerabilities (7 low, 19 moderate, 8 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm warn allow-scripts 3 packages have install scripts not yet covered by allowScripts:
npm warn allow-scripts   @scarf/scarf@1.4.0 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.27.7 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.25.12 (install: (install scripts present))
npm warn allow-scripts
npm warn allow-scripts Run `npm approve-scripts --allow-scripts-pending` to review, or `npm approve-scripts <pkg>` to allow.
✓ Frontend dependencies installed
✓ Google API key loaded


## 🤖 Xây dựng agent

[A2UI](https://a2ui.org/) (Agent-to-UI) là một đặc tả do Google tạo ra dành cho GenUI khai báo. CopilotKit và AG-UI hỗ trợ A2UI một cách nguyên bản - CopilotKit duy trì trình kết xuất React cho A2UI được sử dụng trong bài học này.

### 1. Khởi động server backend

Khởi động một máy chủ FastAPI với một agent graph dạng giữ chỗ (placeholder), tương tự các bài học trước:

In [ ]:
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent
from fastapi import FastAPI
from helper import start_server

app = FastAPI()

# Tạo agent với đồ thị giữ chỗ (sẽ được cập nhật ở ô code sau)
graph = create_agent("google_genai:gemini-3.5-flash-lite")
agent = LangGraphAGUIAgent(
    name="demo_agent",
    description="Demo agent",
    graph=graph
)

add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Khởi chạy server ở port 8004
start_server(app, port=8004)

✓ Server running at http://localhost:8004


### 2. Định nghĩa agent và công cụ

Tại đây, bạn sẽ kích hoạt khả năng sinh A2UI thông qua `CopilotKitMiddleware`. Khi A2UI được bật, CopilotKit tự động thêm các hành vi backend cần thiết để tạo ra cấu trúc A2UI (bạn không cần tự triển khai luồng này).

Bạn cũng sẽ thêm một tool tên là `get_sales_data` - một công cụ lấy dữ liệu truyền thống trả về các số liệu bán hàng. Điều này cho thấy A2UI hoạt động trơn tru cùng với các tool thông thường: Agent gọi dữ liệu trước, sau đó trực quan hóa nó bằng công cụ A2UI được tiêm tự động.

In [3]:
import json
from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver


# Công cụ fetch dữ liệu (đại diện cho việc gọi database/API thực tế)
@tool
def get_sales_data() -> str:
    """Lấy các chỉ số bán hàng và dữ liệu doanh thu hiện tại.

    Trả về dữ liệu bán hàng bao gồm doanh thu, số lượng khách hàng, tỷ lệ chuyển đổi,
    và phân tích chi tiết theo danh mục cũng như theo tháng.
    """
    # Placeholder: Trong thực tế, đoạn này sẽ truy vấn database/API của bạn.
    return json.dumps({
        "totalRevenue": "$1.2M",
        "newCustomers": 3842,
        "conversionRate": "3.6%",
        "revenueByCategory": [
            {"label": "Electronics", "value": 420000},
            {"label": "Clothing", "value": 310000},
            {"label": "Home & Garden", "value": 185000},
            {"label": "Sports", "value": 160000},
            {"label": "Books", "value": 125000},
        ],
        "monthlySales": [
            {"label": "Jan", "value": 85000},
            {"label": "Feb", "value": 92000},
            {"label": "Mar", "value": 108000},
            {"label": "Apr", "value": 95000},
            {"label": "May", "value": 115000},
            {"label": "Jun", "value": 125000},
        ],
    })


graph = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite"),
    tools=[get_sales_data],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "Bạn là một trợ lý ảo hữu ích chuyên tạo giao diện người dùng (UI) trực quan phong phú.\n\n"
        "Hướng dẫn sử dụng công cụ:\n"
        "- Đối với các yêu cầu về dữ liệu kinh doanh/bán hàng: trước tiên hãy gọi get_sales_data để lấy "
        "các chỉ số mới nhất, sau đó gọi generate_a2ui để trực quan hóa kết quả "
        "thành một bảng điều khiển (dashboard) với các biểu đồ, chỉ số và thẻ thông tin.\n"
        "- Đối với các yêu cầu giao diện phong phú khác: gọi trực tiếp generate_a2ui.\n\n"
        "QUAN TRỌNG: Sau khi gọi công cụ, KHÔNG lặp lại hoặc tóm tắt lại dữ liệu "
        "trong câu trả lời bằng văn bản của bạn. Công cụ sẽ tự động hiển thị giao diện. "
        "Chỉ cần xác nhận những gì đã được hiển thị."
    ),
)

agent.graph = graph
print("✓ Đã cập nhật đồ thị agent!")

✓ Đã cập nhật đồ thị agent!


---

## 🎨 Thêm kết xuất A2UI vào frontend

Frontend có hai phần chính: **CopilotKit runtime** endpoint và **danh mục component** hỗ trợ GenUI khai báo.

### 1. CopilotKit runtime

Cấu hình `a2ui: { injectA2UITool: true }` sẽ báo cho runtime tự động tiêm một công cụ A2UI mà agent có thể gọi để sinh UI.

In [4]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const langGraphAgent = new LangGraphHttpAgent({ url: "http://localhost:8004" });

const runtime = new CopilotRuntime({
  agents: { default: langGraphAgent },
  a2ui: { injectA2UITool: true }, // Kích hoạt A2UI
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4004 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4004");
});

Overwriting frontend/server.ts


### 2. Danh mục component - Định nghĩa

Định nghĩa component là bản hợp đồng cho mỗi khối cơ bản trong danh mục, bao gồm: tên (agent dùng để gọi), schema (quy định dữ liệu đầu vào bằng Zod), và mô tả.

In [5]:
%%writefile frontend/src/catalog/definitions.ts

import { z } from "zod";

export const demonstrationCatalogDefinitions = {
  Title: {
    description: "Tiêu đề. Dùng cho tiêu đề phần và tiêu đề trang.",
    props: z.object({
      text: z.string(),
      level: z.string().optional(),
    }),
  },

  Text: {
    description: "Thẻ văn bản. Dùng cho các nhãn, giá trị, chú thích.",
    props: z.object({
      text: z.union([z.string(), z.object({ path: z.string() })]),
      variant: z.enum(["h1", "h2", "h3", "body", "caption"]).optional(),
    }),
  },

  Icon: {
    description: "Biểu tượng Material icon theo tên.",
    props: z.object({
      name: z.string(),
      size: z.number().optional(),
    }),
  },

  Image: {
    description: "Thẻ hình ảnh.",
    props: z.object({
      src: z.union([z.string(), z.object({ path: z.string() })]),
      alt: z.union([z.string(), z.object({ path: z.string() })]).optional(),
      width: z.number().optional(),
      height: z.number().optional(),
    }),
  },

  Divider: {
    description: "Đường phân cách ngang.",
    props: z.object({}),
  },

  Card: {
    description: "Thẻ chứa chung với một vị trí dành cho component con.",
    props: z.object({
      child: z.string().optional(),
    }),
  },

  List: {
    description: "Danh sách các component con. Hỗ trợ hướng ngang hoặc dọc.",
    props: z.object({
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
      direction: z.enum(["horizontal", "vertical"]).optional(),
      gap: z.number().optional(),
    }),
  },

  Tabs: {
    description: "Thẻ chứa dạng tab. Mỗi tab có một nhãn và nội dung con.",
    props: z.object({
      tabs: z.array(z.object({ label: z.string(), child: z.string() })),
    }),
  },

  Row: {
    description: "Thẻ chứa bố cục theo chiều ngang.",
    props: z.object({
      gap: z.number().optional(),
      align: z.string().optional(),
      justify: z.string().optional(),
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
    }),
  },

  Column: {
    description: "Thẻ chứa bố cục theo chiều dọc.",
    props: z.object({
      gap: z.number().optional(),
      align: z.string().optional(),
      children: z.union([
        z.array(z.string()),
        z.object({ componentId: z.string(), path: z.string() }),
      ]),
    }),
  },

  DashboardCard: {
    description:
      "Thẻ bảng điều khiển có tiêu đề và phụ đề tùy chọn. Có một slot 'child' chứa nội dung (biểu đồ, chỉ số, v.v.). Sử dụng 'child' với ID của một component duy nhất.",
    props: z.object({
      title: z.string(),
      subtitle: z.string().optional(),
      child: z.string().optional(),
    }),
  },

  Metric: {
    description:
      "Hiển thị chỉ số quan trọng gồm nhãn, giá trị và chỉ số xu hướng tùy chọn. Rất thích hợp cho KPI và số liệu thống kê.",
    props: z.object({
      label: z.string(),
      value: z.string(),
      trend: z.enum(["up", "down", "neutral"]).optional(),
      trendValue: z.string().optional(),
    }),
  },

  PieChart: {
    description:
      "Biểu đồ hình tròn/donut. Cung cấp dữ liệu dưới dạng mảng các đối tượng {label, value, color}.",
    props: z.object({
      data: z.array(
        z.object({
          label: z.string(),
          value: z.number(),
          color: z.string().optional(),
        }),
      ),
      innerRadius: z.number().optional(),
    }),
  },

  BarChart: {
    description:
      "Biểu đồ cột. Cung cấp dữ liệu dưới dạng mảng các đối tượng {label, value}.",
    props: z.object({
      data: z.array(z.object({ label: z.string(), value: z.number() })),
      color: z.string().optional(),
    }),
  },

  Badge: {
    description:
      "Huy hiệu/thẻ trạng thái nhỏ. Dùng cho nhãn, trạng thái, danh mục.",
    props: z.object({
      text: z.string(),
      variant: z
        .enum(["success", "warning", "error", "info", "neutral"])
        .optional(),
    }),
  },

  DataTable: {
    description: "Bảng dữ liệu gồm các cột và dòng.",
    props: z.object({
      columns: z.array(z.object({ key: z.string(), label: z.string() })),
      rows: z.array(z.record(z.any())),
    }),
  },

  Button: {
    description:
      "Nút tương tác. Sử dụng 'label' cho văn bản đơn giản hoặc 'child' cho component con. 'action' sẽ được gửi đi khi nhấp vào.",
    props: z.object({
      label: z.string().optional(),
      child: z
        .string()
        .describe(
          "ID của component con (ví dụ: một Text component làm nhãn).",
        )
        .optional(),
      variant: z.enum(["primary", "secondary", "ghost"]).optional(),
      action: z
        .union([
          z.object({
            event: z.object({
              name: z.string(),
              context: z.record(z.any()).optional(),
            }),
          }),
          z.null(),
        ])
        .optional(),
    }),
  },
};

/** Hỗ trợ kiểu dữ liệu (type helper) cho các renderer */
export type DemonstrationCatalogDefinitions = typeof demonstrationCatalogDefinitions;

Overwriting frontend/src/catalog/definitions.ts


### 3. Danh mục component - Renderer

Renderer là các cài đặt theo từng nền tảng cụ thể để biến các "định nghĩa" thành UI thực sự (React component).

In [6]:
%%writefile frontend/src/catalog/renderers.tsx

import React from "react";
import { PieChart as RechartsPie, Pie, Cell, ResponsiveContainer, BarChart as RechartsBar, Bar, XAxis, YAxis, Tooltip, CartesianGrid } from "recharts";
import { createCatalog, type CatalogRenderers } from "@copilotkit/a2ui-renderer";
import { demonstrationCatalogDefinitions, type DemonstrationCatalogDefinitions } from "./definitions";

// Hàm xử lý dữ liệu động hoặc chuỗi tĩnh

function resolveText(value: unknown): string {
  if (typeof value === "string") return value;
  if (value && typeof value === "object" && "path" in value)
    return String((value as { path: string }).path);
  return String(value ?? "");
}

// Map các thành phần vào React UI

const demonstrationCatalogRenderers: CatalogRenderers<DemonstrationCatalogDefinitions> =
  {
    Title: ({ props }) => {
      const Tag = (
        props.level === "h1" ? "h1" : props.level === "h3" ? "h3" : "h2"
      ) as "h1" | "h2" | "h3";
      const sizes: Record<string, string> = {
        h1: "1.75rem",
        h2: "1.25rem",
        h3: "1rem",
      };
      return (
        <Tag
          style={{
            margin: 0,
            fontWeight: 600,
            fontSize: sizes[props.level ?? "h2"],
            color: "#111827",
            letterSpacing: "-0.01em",
          }}
        >
          {resolveText(props.text)}
        </Tag>
      );
    },

    Text: ({ props }) => {
      const styles: Record<string, React.CSSProperties> = {
        h1: { fontSize: "1.5rem", fontWeight: 700, color: "#111827" },
        h2: { fontSize: "1.25rem", fontWeight: 700, color: "#111827" },
        h3: { fontSize: "1rem", fontWeight: 600, color: "#374151" },
        body: { fontSize: "0.875rem", color: "#374151" },
        caption: { fontSize: "0.75rem", color: "#6b7280" },
      };
      return (
        <span style={styles[props.variant ?? "body"]}>
          {resolveText(props.text)}
        </span>
      );
    },

    Icon: ({ props }) => (
      <span
        className="material-symbols-outlined"
        style={{ fontSize: props.size ?? 24, color: "#6b7280" }}
      >
        {props.name}
      </span>
    ),

    Image: ({ props }) => (
      <img
        src={resolveText(props.src)}
        alt={resolveText(props.alt ?? "")}
        style={{
          width: props.width ?? "auto",
          height: props.height ?? "auto",
          maxWidth: "100%",
          borderRadius: 8,
        }}
      />
    ),

    Divider: () => (
      <hr style={{ border: "none", borderTop: "1px solid #e5e7eb", margin: "4px 0" }} />
    ),

    Card: ({ props, children }) => (
      <div
        style={{
          background: "#fff",
          borderRadius: 12,
          border: "1px solid #e5e7eb",
          padding: 16,
          boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
        }}
      >
        {typeof props.child === "string" && children(props.child)}
      </div>
    ),

    List: ({ props, children }) => {
      const items = Array.isArray(props.children) ? props.children : [];
      const isHorizontal = (props as any).direction === "horizontal";
      return (
        <div
          style={{
            display: "flex",
            flexDirection: isHorizontal ? "row" : "column",
            gap: props.gap ?? 8,
            overflowX: isHorizontal ? "auto" : undefined,
            flexWrap: isHorizontal ? "nowrap" : undefined,
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return <React.Fragment key={`${item}-${i}`}>{children(item)}</React.Fragment>;
            if (item && typeof item === "object" && "id" in item)
              return (
                <div key={`${item.id}-${i}`} style={isHorizontal ? { flex: "0 0 auto", minWidth: 280 } : undefined}>
                  {(children as any)(item.id, item.basePath)}
                </div>
              );
            return null;
          })}
        </div>
      );
    },

    Tabs: ({ props, children }) => {
      const [active, setActive] = React.useState(0);
      const tabs = props.tabs ?? [];
      return (
        <div>
          <div style={{ display: "flex", gap: 0, borderBottom: "1px solid #e5e7eb" }}>
            {tabs.map((tab: any, i: number) => (
              <button
                key={i}
                onClick={() => setActive(i)}
                style={{
                  padding: "8px 16px",
                  fontSize: "0.85rem",
                  fontWeight: active === i ? 600 : 400,
                  color: active === i ? "#111827" : "#6b7280",
                  borderBottom: active === i ? "2px solid #111827" : "2px solid transparent",
                  background: "none",
                  border: "none",
                  cursor: "pointer",
                }}
              >
                {tab.label}
              </button>
            ))}
          </div>
          <div style={{ padding: "12px 0" }}>
            {tabs[active] && children(tabs[active].child)}
          </div>
        </div>
      );
    },

    Row: ({ props, children }) => {
      const justifyMap: Record<string, string> = {
        start: "flex-start",
        center: "center",
        end: "flex-end",
        spaceBetween: "space-between",
      };
      const items = Array.isArray(props.children) ? props.children : [];
      return (
        <div
          style={{
            display: "flex",
            flexDirection: "row",
            gap: `${props.gap ?? 16}px`,
            alignItems: props.align ?? "stretch",
            justifyContent:
              justifyMap[props.justify ?? "start"] ?? "flex-start",
            flexWrap: "wrap",
            width: "100%",
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return (
                <div
                  key={`${item}-${i}`}
                  style={{ flex: "1 1 0", minWidth: 0 }}
                >
                  {children(item)}
                </div>
              );
            if (item && typeof item === "object" && "id" in item)
              return (
                <div
                  key={`${item.id}-${i}`}
                  style={{ flex: "1 1 0", minWidth: 0 }}
                >
                  {(children as any)(item.id, item.basePath)}
                </div>
              );
            return null;
          })}
        </div>
      );
    },

    Column: ({ props, children }) => {
      const items = Array.isArray(props.children) ? props.children : [];
      return (
        <div
          style={{
            display: "flex",
            flexDirection: "column",
            gap: `${props.gap ?? 12}px`,
            width: "100%",
          }}
        >
          {items.map((item: any, i: number) => {
            if (typeof item === "string")
              return (
                <React.Fragment key={`${item}-${i}`}>
                  {children(item)}
                </React.Fragment>
              );
            if (item && typeof item === "object" && "id" in item)
              return (
                <React.Fragment key={`${item.id}-${i}`}>
                  {(children as any)(item.id, item.basePath)}
                </React.Fragment>
              );
            return null;
          })}
        </div>
      );
    },

    DashboardCard: ({ props, children }) => (
      <div
        style={{
          background: "#fff",
          borderRadius: "12px",
          border: "1px solid #e5e7eb",
          padding: "20px",
          boxShadow: "0 1px 3px rgba(0,0,0,0.04)",
          display: "flex",
          flexDirection: "column",
          gap: "12px",
        }}
      >
        <div>
          <div
            style={{ fontWeight: 600, fontSize: "0.9rem", color: "#111827" }}
          >
            {resolveText(props.title)}
          </div>
          {props.subtitle && (
            <div
              style={{
                fontSize: "0.75rem",
                color: "#6b7280",
                marginTop: "2px",
              }}
            >
              {resolveText(props.subtitle)}
            </div>
          )}
        </div>
        {typeof props.child === "string" && children(props.child)}
      </div>
    ),

    Metric: ({ props }) => {
      const trendColors: Record<string, string> = {
        up: "#059669",
        down: "#dc2626",
        neutral: "#6b7280",
      };
      const trendIcons: Record<string, string> = {
        up: "↑",
        down: "↓",
        neutral: "→",
      };
      return (
        <div style={{ display: "flex", flexDirection: "column", gap: "4px" }}>
          <span
            style={{
              fontSize: "0.75rem",
              color: "#6b7280",
              fontWeight: 500,
              textTransform: "uppercase",
              letterSpacing: "0.05em",
            }}
          >
            {resolveText(props.label)}
          </span>
          <div style={{ display: "flex", alignItems: "baseline", gap: "8px" }}>
            <span
              style={{
                fontSize: "1.5rem",
                fontWeight: 700,
                color: "#111827",
                letterSpacing: "-0.02em",
              }}
            >
              {resolveText(props.value)}
            </span>
            {props.trend && props.trendValue && (
              <span
                style={{
                  fontSize: "0.8rem",
                  fontWeight: 500,
                  color: trendColors[props.trend] ?? "#6b7280",
                }}
              >
                {trendIcons[props.trend]} {resolveText(props.trendValue)}
              </span>
            )}
          </div>
        </div>
      );
    },

    PieChart: ({ props }) => {
      const COLORS = [
        "#3b82f6",
        "#8b5cf6",
        "#ec4899",
        "#f59e0b",
        "#10b981",
        "#6366f1",
      ];
      const data = props.data ?? [];
      return (
        <div style={{ width: "100%", height: 200 }}>
          <ResponsiveContainer>
            <RechartsPie>
              <Pie
                data={data}
                dataKey="value"
                nameKey="label"
                cx="50%"
                cy="50%"
                innerRadius={props.innerRadius ?? 40}
                outerRadius={80}
                paddingAngle={2}
              >
                {data.map((entry: any, i: number) => (
                  <Cell
                    key={i}
                    fill={entry.color ?? COLORS[i % COLORS.length]}
                  />
                ))}
              </Pie>
              <Tooltip />
            </RechartsPie>
          </ResponsiveContainer>
        </div>
      );
    },

    BarChart: ({ props }) => {
      const data = props.data ?? [];
      return (
        <div style={{ width: "100%", height: 200 }}>
          <ResponsiveContainer>
            <RechartsBar data={data}>
              <CartesianGrid strokeDasharray="3 3" stroke="#f3f4f6" />
              <XAxis dataKey="label" tick={{ fontSize: 11, fill: "#6b7280" }} />
              <YAxis tick={{ fontSize: 11, fill: "#6b7280" }} />
              <Tooltip />
              <Bar
                dataKey="value"
                fill={props.color ?? "#3b82f6"}
                radius={[4, 4, 0, 0]}
              />
            </RechartsBar>
          </ResponsiveContainer>
        </div>
      );
    },

    Badge: ({ props }) => {
      const variants: Record<string, { bg: string; color: string }> = {
        success: { bg: "#dcfce7", color: "#166534" },
        warning: { bg: "#fef3c7", color: "#92400e" },
        error: { bg: "#fee2e2", color: "#991b1b" },
        info: { bg: "#dbeafe", color: "#1e40af" },
        neutral: { bg: "#f3f4f6", color: "#374151" },
      };
      const v = variants[props.variant ?? "neutral"] ?? variants.neutral;
      return (
        <span
          style={{
            display: "inline-block",
            padding: "2px 8px",
            borderRadius: "9999px",
            fontSize: "0.7rem",
            fontWeight: 500,
            background: v.bg,
            color: v.color,
          }}
        >
          {resolveText(props.text)}
        </span>
      );
    },

    DataTable: ({ props }) => {
      const cols = props.columns ?? [];
      const rows = props.rows ?? [];
      return (
        <div style={{ overflowX: "auto", width: "100%" }}>
          <table
            style={{
              width: "100%",
              borderCollapse: "collapse",
              fontSize: "0.8rem",
            }}
          >
            <thead>
              <tr>
                {cols.map((col: any) => (
                  <th
                    key={col.key}
                    style={{
                      textAlign: "left",
                      padding: "8px 12px",
                      borderBottom: "2px solid #e5e7eb",
                      color: "#6b7280",
                      fontWeight: 600,
                      fontSize: "0.7rem",
                      textTransform: "uppercase",
                      letterSpacing: "0.05em",
                    }}
                  >
                    {col.label}
                  </th>
                ))}
              </tr>
            </thead>
            <tbody>
              {rows.map((row: any, i: number) => (
                <tr key={i} style={{ borderBottom: "1px solid #f3f4f6" }}>
                  {cols.map((col: any) => (
                    <td
                      key={col.key}
                      style={{ padding: "8px 12px", color: "#374151" }}
                    >
                      {String(row[col.key] ?? "")}
                    </td>
                  ))}
                </tr>
              ))}
            </tbody>
          </table>
        </div>
      );
    },

    Button: ({ props, children, dispatch }) => {
      const variants: Record<string, React.CSSProperties> = {
        primary: { background: "#111827", color: "#fff", border: "none" },
        secondary: {
          background: "#fff",
          color: "#374151",
          border: "1px solid #d1d5db",
        },
        ghost: { background: "transparent", color: "#3b82f6", border: "none" },
      };
      const style = variants[props.variant ?? "primary"] ?? variants.primary;
      return (
        <button
          style={{
            ...style,
            padding: "8px 16px",
            borderRadius: "8px",
            fontSize: "0.8rem",
            fontWeight: 500,
            cursor: "pointer",
            transition: "opacity 0.15s",
            width: "100%",
          }}
          onClick={() => dispatch?.(props.action)}
        >
          {typeof props.child === "string" ? children(props.child) : (props as any).label ?? null}
        </button>
      );
    },
  };

// Khởi tạo catalog hoàn chỉnh

export const demonstrationCatalog = createCatalog(
  demonstrationCatalogDefinitions,
  demonstrationCatalogRenderers,
  {
    catalogId: "copilotkit://app-dashboard-catalog",
    includeBasicCatalog: false,
  },
);

Overwriting frontend/src/catalog/renderers.tsx


### 4. Kết nối mọi thứ - `main.tsx` và `App.tsx`

Đăng ký danh mục vào `CopilotKitProvider`.

In [7]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import { demonstrationCatalog } from "./catalog/renderers";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit
        useSingleEndpoint={false}
        runtimeUrl="/api/copilotkit"

        // 🪁 Kích hoạt A2UI và nạp danh mục vừa tạo
        a2ui={{ catalog: demonstrationCatalog }}
      >
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

Overwriting frontend/src/main.tsx


Trong `App.tsx`, ta thêm các gợi ý cho khung chat:

In [8]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";
import { useExampleDynamicSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {
  useExampleDynamicSuggestions(); // Thêm gợi ý chat "Sales Dashboard"
  return <CopilotChat />;
}

Overwriting frontend/src/App.tsx


Khởi động và hiển thị frontend:

In [ ]:
# Khởi động frontend ở port 3004
from helper import start_frontend, display_app

start_frontend(port=3004)

# Hiển thị UI ngay trên Jupyter Notebook
display_app(port=3004)

---

## 🎯 Sinh UI với schema tĩnh

Cho đến giờ, bạn đã xây dựng một phiên bản **schema động** - nơi agent tự định nghĩa cấu trúc cây A2UI ngay lúc chạy. Nhưng có một giải pháp khác: **Schema tĩnh**.

Với schema cố định, bạn thiết kế trước bộ khung UI - bố cục, cách lồng ghép, và đường dẫn dữ liệu. Công việc duy nhất của agent là gọi dữ liệu để lấp đầy khung đó. Cách này đem lại sự kiểm soát tuyệt đối về thiết kế mà vẫn giữ được sự thông minh của AI.

Rất hữu ích cho các giao diện cần sự trau chuốt, bóng bẩy (như thẻ chuyến bay, hóa đơn).

### Sử dụng A2UI Composer

Cách dễ nhất để xây dựng một schema cố định là dùng công cụ thiết kế [A2UI Composer](https://a2ui-editor.ag-ui.com/).

<img src="images/a2ui-composer.png" style="width: 75%; display: block; margin: 0 auto;">

Bạn có thể nhập prompt: *"Hãy tạo carousel thẻ chuyến bay hiển thị nơi đi, nơi đến, thời lượng bay, giờ đi và giờ đến."* để hệ thống sinh ra cấu trúc, sau đó bấm `Copy JSON` để đưa vào mã nguồn.

<img src="images/a2ui-composer-schema.png" style="width: 75%; display: block; margin: 0 auto;">

### Thêm công cụ schema tĩnh

Trong AG-UI, **bất kỳ công cụ nào** cũng có thể trả về các hành động A2UI, chỉ cần bạn bọc kết quả trong cấu trúc mảng `a2ui_operations`.

Dưới đây là cách định nghĩa một công cụ trả về schema cố định (được tạo bởi Composer) và tiêm dữ liệu bay vào:

In [10]:
from typing_extensions import TypedDict
from copilotkit import a2ui
from langchain.tools import tool

CATALOG_ID = "copilotkit://app-dashboard-catalog"
SURFACE_ID = "flight-search-results"

FLIGHT_SCHEMA = [
    {"id": "root", "component": "List", "children": {"componentId": "flight-card", "path": "/flights"}, "direction": "horizontal", "gap": 16},
    {"id": "flight-card", "component": "Card", "child": "main-col"},
    {"id": "main-col", "component": "Column", "children": ["airline-img", "header-row", "meta-row", "divider-1", "times-row", "route-row", "divider-2", "status-row", "divider-3", "book-btn"], "align": "stretch", "gap": 8},
    {"id": "airline-img", "component": "Image", "src": {"path": "airlineLogo"}, "alt": {"path": "airline"}, "height": 32},
    {"id": "header-row", "component": "Row", "children": ["airline-name", "price-text"], "justify": "spaceBetween", "align": "center"},
    {"id": "airline-name", "component": "Text", "text": {"path": "airline"}, "variant": "h3"},
    {"id": "price-text", "component": "Text", "text": {"path": "price"}, "variant": "h2"},
    {"id": "meta-row", "component": "Row", "children": ["flight-number", "date-text"], "justify": "spaceBetween", "align": "center"},
    {"id": "flight-number", "component": "Text", "text": {"path": "flightNumber"}, "variant": "caption"},
    {"id": "date-text", "component": "Text", "text": {"path": "date"}, "variant": "caption"},
    {"id": "divider-1", "component": "Divider"},
    {"id": "times-row", "component": "Row", "children": ["depart-time", "duration-text", "arrive-time"], "justify": "spaceBetween", "align": "center"},
    {"id": "depart-time", "component": "Text", "text": {"path": "departureTime"}, "variant": "h2"},
    {"id": "duration-text", "component": "Text", "text": {"path": "duration"}, "variant": "caption"},
    {"id": "arrive-time", "component": "Text", "text": {"path": "arrivalTime"}, "variant": "h2"},
    {"id": "route-row", "component": "Row", "children": ["origin-code", "arrow-text", "dest-code"], "justify": "spaceBetween", "align": "center"},
    {"id": "origin-code", "component": "Text", "text": {"path": "origin"}, "variant": "h3"},
    {"id": "arrow-text", "component": "Text", "text": "\u2192", "variant": "h3"},
    {"id": "dest-code", "component": "Text", "text": {"path": "destination"}, "variant": "h3"},
    {"id": "divider-2", "component": "Divider"},
    {"id": "status-row", "component": "Row", "children": ["status-text"], "align": "center"},
    {"id": "status-text", "component": "Text", "text": {"path": "status"}, "variant": "caption"},
    {"id": "divider-3", "component": "Divider"},
    {"id": "book-btn", "component": "Button", "label": "Book Flight", "variant": "primary", "action": {"event": {"name": "bookFlight"}}},
]


class Flight(TypedDict):
    id: str
    airline: str
    airlineLogo: str
    flightNumber: str
    origin: str
    destination: str
    date: str
    departureTime: str
    arrivalTime: str
    duration: str
    status: str
    price: str


# Công cụ lấy dữ liệu (đại diện cho một API tìm kiếm chuyến bay thực tế)
@tool
def search_flights(origin: str, destination: str) -> list[Flight]:
    """Tìm kiếm các chuyến bay khả dụng giữa hai sân bay.

    Args:
        origin: Mã IATA sân bay đi (ví dụ: "SFO").
        destination: Mã IATA sân bay đến (ví dụ: "JFK").
    """
    # Dữ liệu mẫu: Trong môi trường thực tế, đoạn này sẽ gọi một API tìm kiếm chuyến bay thực.
    return [
        {"id": "1", "airline": "Delta Air Lines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/DL.png", "flightNumber": "DL 520", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "08:00", "arrivalTime": "16:35", "duration": "5h 35m", "status": "On Time", "price": "$389"},
        {"id": "2", "airline": "United Airlines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/UA.png", "flightNumber": "UA 1583", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "10:15", "arrivalTime": "18:42", "duration": "5h 27m", "status": "On Time", "price": "$412"},
        {"id": "3", "airline": "JetBlue", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/B6.png", "flightNumber": "B6 416", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "14:30", "arrivalTime": "23:05", "duration": "5h 35m", "status": "On Time", "price": "$345"},
        {"id": "4", "airline": "American Airlines", "airlineLogo": f"https://www.gstatic.com/flights/airline_logos/70px/AA.png", "flightNumber": "AA 178", "origin": origin, "destination": destination, "date": "2026-04-11", "departureTime": "17:00", "arrivalTime": "01:20+1", "duration": "5h 20m", "status": "On Time", "price": "$398"},
    ]


@tool
def display_flights(flights: list[Flight]) -> str:
    """Hiển thị các chuyến bay dưới dạng thẻ trực quan theo hàng ngang.

    Mỗi chuyến bay phải bao gồm: id, airline, airlineLogo (đường dẫn URL), flightNumber,
    origin, destination, date, departureTime, arrivalTime, duration,
    status, và price.
    """
    return a2ui.render(
        operations=[
            a2ui.create_surface(SURFACE_ID, catalog_id=CATALOG_ID),
            a2ui.update_components(SURFACE_ID, FLIGHT_SCHEMA),
            a2ui.update_data_model(SURFACE_ID, {"flights": flights}),
        ],
    )

Cập nhật lại đồ thị agent để bao gồm cả công cụ động (`generate_a2ui`) và tĩnh (`display_flights`), giúp AI linh hoạt lựa chọn:

In [ ]:
# Tái tạo agent graph với cả hai công cụ: động (generate_a2ui) + cố định (search_flights)
graph = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite"),
    tools=[get_sales_data, search_flights, display_flights],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "Bạn là một trợ lý hữu ích chuyên tạo các giao diện người dùng (UI) trực quan phong phú.\n\n"
        "Hướng dẫn sử dụng công cụ:\n"
        "- TẤT CẢ các truy vấn liên quan đến chuyến bay: trước tiên hãy gọi search_flights để lấy "
        "dữ liệu chuyến bay, sau đó gọi display_flights với kết quả thu được. KHÔNG BAO GIỜ dùng generate_a2ui "
        "cho thông tin chuyến bay.\n"
        "- Đối với các yêu cầu dữ liệu kinh doanh/bán hàng: trước tiên gọi get_sales_data để lấy "
        "các chỉ số mới nhất, sau đó gọi generate_a2ui để trực quan hóa kết quả.\n"
        "- Đối với các yêu cầu giao diện phong phú khác: gọi trực tiếp generate_a2ui.\n\n"
        "Logo các hãng hàng không: sử dụng định dạng https://www.gstatic.com/flights/airline_logos/70px/<MÃ_IATA>.png\n"
        "Các mã IATA phổ biến: DL=Delta, UA=United, AA=American, WN=Southwest, B6=JetBlue, "
        "NK=Spirit, AS=Alaska, F9=Frontier, BA=British Airways, LH=Lufthansa, "
        "AF=Air France, EK=Emirates, QF=Qantas, SQ=Singapore Airlines, NH=ANA.\n\n"
        "QUAN TRỌNG: Sau khi gọi một công cụ, KHÔNG lặp lại hoặc tóm tắt lại dữ liệu "
        "trong câu phản hồi bằng văn bản của bạn. Công cụ sẽ tự động hiển thị giao diện. "
        "Chỉ cần xác nhận những gì đã được hiển thị."
    ),
)

agent.graph = graph
print("✓ Đã cập nhật đồ thị agent với công cụ display_flights!")

✓ Đã cập nhật đồ thị agent với công cụ display_flights!


Trong `App.tsx`, ta thêm các gợi ý cho khung chat:

In [12]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";
import { 
  useExampleDynamicSuggestions,
  useExampleFixedSuggestions
} from "@/hooks/use-example-suggestions";

export default function App() {
  useExampleDynamicSuggestions();
  useExampleFixedSuggestions();

  return <CopilotChat />;
}

Overwriting frontend/src/App.tsx


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/.venv/lib/python3.12/site-packages/uvicorn/protocols/http/h11_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/.venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/.venv/lib/python3.12/site-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/.venv/lib/python3.12/site-packages/starlette/applications.py", line 90, in __cal

Khi chạy và yêu cầu hiển thị chuyến bay, đây là kết quả của schema cố định:

<img src="images/a2ui-fixed-carousel.png" style="width: 75%; display: block; margin: 0 auto;">

---

## 🆚 Schema động vs schema tĩnh: Khi nào dùng loại nào?

| Yếu tố | **Schema tĩnh** | **Schema động** |
|---|---|---|
| **Bố cục** | Cố định, giống hệt nhau mọi lần gọi | Do agent sinh ra, thay đổi theo mỗi yêu cầu |
| **Vai trò của agent** | Chỉ điền dữ liệu | Chọn component và tự thiết kế bố cục |
| **Tính nhất quán** | Tuyệt đối | Có thể thay đổi |
| **Sự linh hoạt** | Tối thiểu - Đổi layout cần sửa code | Cao - Agent tự điều chỉnh theo prompt |
| **Tốt nhất cho** | UI có lưu lượng truy cập cao, yêu cầu chuẩn thương hiệu (Thẻ chuyến bay, hóa đơn) | UI "đuôi dài", khám phá dữ liệu hoặc công cụ nội bộ |

*Trong thực tế, nhiều ứng dụng sử dụng kết hợp cả hai: Schema tĩnh cho các bề mặt quan trọng, và schema động cho mọi thứ khác.*

---

## 🎓 Tổng kết những gì bạn đã học

- **GenUI khai báo** cho phép agent lắp ráp giao diện từ danh mục các khối xây dựng - linh hoạt hơn GenUI kiểm soát, nhưng nhất quán hơn GenUI mở.
- Đặc tả **A2UI** bao gồm 3 yếu tố: **Danh mục component** (định nghĩa + renderer), **schema** (cách sắp xếp), và **dữ liệu truyền vào**.
- **Schema động** cho phép agent tự tạo bố cục ngay trong lúc chạy - tuyệt vời cho các tình huống truy vấn mở.
- **Schema tĩnh** cung cấp một bố cục tĩnh được lập trình trước và agent chỉ việc đổ dữ liệu vào - hoàn hảo cho các UI trau chuốt, dùng nhiều lần.
- Cả hai phương pháp đều có thể cùng tồn tại và hoạt động mượt mà trong cùng một agent.

---

## ⏭️ Bước tiếp theo

Trong bài tiếp theo, bạn sẽ tiến đến một cấp độ mới: **GenUI mở** - cực hạn của sự linh hoạt. Bạn sẽ kết nối một MCP app (Excalidraw) để agent có thể khởi chạy toàn bộ ứng dụng ngay trong khung chat, và bật tính năng `openGenerativeUI` cho phép agent sinh ra UI lập trình một cách tự do.